# Integer GPT — free Colab training

This notebook trains the resumable Solid 2 tiny GPT. Training uses
floating point; the exported checkpoint is quantized to int8/Q16 and
runs integer-only in the browser and WebAssembly demo.

A free T4 session is enough. The trainer checkpoints to Google Drive,
so when the session ends you run the same command with `--resume` and
continue from the same optimizer step.

Before you start, open **Runtime → Change runtime type → T4 GPU**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/tiny-gpt-solid2'
print('Checkpoints go to', OUT)


In [ ]:
import os

REPO = '/content/only-gates-ai'
if not os.path.isdir(REPO):
    !git clone https://github.com/clinuxrulz/only-gates-ai.git {REPO}
%cd {REPO}
!git pull --ff-only || true


## 1. Smoke test

This proves the corpus, tokenizer, model, and checkpoint path work in
under a minute. It does not need a GPU.


In [ ]:
!python scripts/train_tiny_gpt_solid2_colab.py \
  --out /content/tiny-gpt-smoke --smoke --fresh


## 2. Train or resume

The same cell works for a new run and for a later session. With
`--resume`, the trainer loads `latest.pt` if it exists; otherwise it
starts fresh. It also verifies the configuration, seed, tokenizer, and
data fingerprint before continuing.

Use `--fresh` only when you intentionally changed the configuration or
data.


In [ ]:
# Same command for the first session and every later session.
# Add --fresh if you changed the model config or data and want a new run.
!python scripts/train_tiny_gpt_solid2_colab.py \
  --out {OUT} \
  --lm-steps 5000 --qa-epochs 100 \
  --save-every 100 --max-minutes 300 \
  --resume


## 3. Inspect the run

`run.json` records the stage, fingerprint, resume count, and wall time.
`metrics.jsonl` is append-only, so it survives disconnects.


In [ ]:
import json
from pathlib import Path

out = Path(OUT)
run = out / 'run.json'
if run.exists():
    print(run.read_text())

metrics = out / 'metrics.jsonl'
if metrics.exists():
    lines = metrics.read_text().strip().splitlines()
    print('metrics lines:', len(lines))
    for line in lines[-5:]:
        print(line)


## 4. Hash and share the artifacts

Report the checkpoint hash with your run. Do not paste large binaries
into an issue; link a release asset, Hugging Face, or an accessible
Drive folder.


In [ ]:
import hashlib
from pathlib import Path

out = Path(OUT)
latest = out / 'latest.pt'
if latest.exists():
    digest = hashlib.sha256(latest.read_bytes()).hexdigest()
    print('latest.pt sha256:', digest)
    print('files:', sorted(p.name for p in out.glob('*') if p.is_file()))
else:
    print('No checkpoint yet. Run the training cell first.')


In [ ]:
# Optional: bundle the run for download.
import shutil
from pathlib import Path
from google.colab import files

out = Path(OUT)
archive = shutil.make_archive('/content/tiny-gpt-solid2-artifacts', 'zip', root_dir=out)
print('wrote', archive)
# files.download(archive)


## 5. Report the run

Open a **Training result** issue with the hash, command, seed,
configuration, hardware, wall time, session count, and metrics:

https://github.com/clinuxrulz/only-gates-ai/issues/new?template=training-result.yml

Read `docs/COMPUTE.md` for the full protocol and troubleshooting.
Follow Colab's terms of service, keep one free account, and stop the
runtime when you are done.
